# critique_questions — research notebook

The quality gate before the quiz ships: a second LLM pass that checks every question **against the article**, one verdict per question. Failures route back through `repair_questions` (same generator, failed slots only, with this feedback) for at most 2 rounds.

Checks per question: **grounded** (answer derivable from the article alone), **unambiguous** (exactly one defensible correct option), **distractors** (plausible but clearly wrong), **level** (actually tests the assigned Bloom level).

```
critique_questions(clean_text, questions) -> CritiqueResult(ok, verdicts, error)
```

**Resilience rule: never raises** — and on critic failure the run ships the questions unreviewed rather than dying (a good-enough quiz beats no quiz).

In [ ]:
import os
from typing import TypedDict
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(".env")

class QuizQuestion(TypedDict):
    slot_id: int
    question: str
    options: list[str]
    correct_answer: int
    explanation: str

class Verdict(TypedDict):
    slot_id: int
    passed: bool
    reason: str              # short; becomes repair feedback when failed

class VerdictBatch(TypedDict):
    verdicts: list[Verdict]

class CritiqueResult(TypedDict):
    ok: bool
    verdicts: list[Verdict]
    error: str

llm = ChatOpenAI(model=os.getenv("LLM_MODEL", "gpt-5-mini"))
critic = llm.with_structured_output(VerdictBatch)

## The node

The critic never sees which option the generator marked correct until after judging — it must find the correct answer itself from the article; disagreement is an automatic fail. That catches wrong-answer-key bugs, the worst failure mode a quiz can have.

In [ ]:
SYSTEM = """You are reviewing quiz questions against the article they were generated from.
For each question, FIRST determine the correct option yourself using ONLY the article.
Then give a verdict. FAIL a question if any of these hold:
- your answer differs from the marked correct_answer (wrong answer key)
- the answer cannot be determined from the article alone (ungrounded)
- more than one option is defensible (ambiguous)
- a distractor is absurd or trivially eliminable (weak distractors)
Otherwise PASS. Keep reasons to one short sentence; for fails, say exactly what to fix."""

def critique_questions(clean_text: str, questions: list[QuizQuestion]) -> CritiqueResult:
    """Never raises: on failure, returns ok=False (graph ships unreviewed rather than dying)."""
    try:
        qlines = "\n\n".join(
            f'slot_id {q["slot_id"]}: {q["question"]}\n' +
            "\n".join(f"  option {i}: {o}" for i, o in enumerate(q["options"])) +
            f'\n  marked correct_answer: {q["correct_answer"]}'
            for q in questions)
        out: VerdictBatch = critic.invoke([
            ("system", SYSTEM),
            ("user", f"Article:\n{clean_text}\n\nQuestions:\n{qlines}"),
        ])
        # verdict for every question, defaulting missing ones to pass (fail-open)
        got = {v["slot_id"]: v for v in out["verdicts"]}
        verdicts = [got.get(q["slot_id"],
                    Verdict(slot_id=q["slot_id"], passed=True, reason="no verdict returned"))
                    for q in questions]
        return CritiqueResult(ok=True, verdicts=verdicts, error="")
    except Exception as e:                       # noqa: BLE001 — resilience boundary
        return CritiqueResult(ok=False, verdicts=[], error=f"Critique failed: {e}")

def failed_feedback(verdicts: list[Verdict]) -> str:
    """Feedback string for repair_questions."""
    return "\n".join(f'slot {v["slot_id"]}: {v["reason"]}' for v in verdicts if not v["passed"])

## Live test: one good question, one sabotaged

The sabotaged question has a deliberately wrong answer key — the critic should catch it by solving the question itself.

In [ ]:
ARTICLE = (
    "The transformer replaces recurrence with self-attention, letting every token attend to "
    "every other token in one step. Multi-head attention runs several attention functions in "
    "parallel so different heads can focus on different relationships, such as syntax versus "
    "meaning. Because the model sees tokens as an unordered set, positional encodings are "
    "added to the embeddings so word order still matters."
)
questions: list[QuizQuestion] = [
    {"slot_id": 0,
     "question": "Why does the transformer add positional encodings to the embeddings?",
     "options": ["Because tokens are seen as an unordered set and word order must still matter",
                 "To let several attention heads run in parallel",
                 "To reintroduce recurrence into the model",
                 "To compress the vocabulary size"],
     "correct_answer": 0, "explanation": "Stated directly in the article."},
    {"slot_id": 1,
     "question": "What does multi-head attention allow the model to do?",
     "options": ["Focus different heads on different relationships",
                 "Process tokens strictly in order",
                 "Remove the need for embeddings",
                 "Guarantee smaller model size"],
     "correct_answer": 2,   # SABOTAGED: correct is 0
     "explanation": "..."},
]
res = critique_questions(ARTICLE, questions)
if res["ok"]:
    for v in res["verdicts"]:
        print(f'slot {v["slot_id"]}: {"PASS" if v["passed"] else "FAIL"} — {v["reason"]}')
    print("\nrepair feedback:\n" + (failed_feedback(res["verdicts"]) or "(none)"))
else:
    print("graceful failure:", res["error"])

In [ ]:
# resilience + helper checks (offline)
assert failed_feedback([{"slot_id": 3, "passed": False, "reason": "wrong key"},
                        {"slot_id": 4, "passed": True, "reason": "ok"}]) == "slot 3: wrong key"
broken = critique_questions(ARTICLE, [{"slot_id": 0}])   # malformed question dict
assert broken["ok"] is False or broken["verdicts"]
print("offline checks passed")

## Notes

- **Answer-key check is the core:** the critic solves each question independently; disagreement fails the slot. This is the check that catches the worst bug a quiz can ship.
- **Fail-open by design:** critic crash or missing verdict → questions ship unreviewed. Quality gate, not a single point of failure. Log it in production.
- The verdict loop in the graph: failures + round < 2 → `repair_questions` with `failed_feedback`; else proceed with passing questions only.
- Same small model works — critique is easier than generation. If quality data later says otherwise, upgrade only this node.
- V2 idea: a user-triggered fact-check button rides this same node with web search added.